# Lakebase 101 — Post-Deploy

Run this **after** `databricks bundle deploy`.

1. Grants the app's service principal `CAN_MANAGE_RUN` on synced-table pipelines
2. Starts and deploys the app

In [0]:
CATALOG = "lakebase_101_catalog"
APP_NAME = "lakebase-101-app"

In [0]:
"""Grant the app's service principal CAN_MANAGE_RUN on synced-table pipelines.
This allows the 'Sync Now' button in the app to trigger on-demand refreshes."""
import time
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.pipelines import PipelineAccessControlRequest, PipelinePermissionLevel

APP_NAME = "lakebase-101-app"

w = WorkspaceClient()

# Wait briefly for the app to be registered after deploy
for attempt in range(5):
    try:
        app = w.apps.get(APP_NAME)
        sp_name = app.service_principal_client_id
        print(f"App SP (client_id): {sp_name}")
        print(f"App SP (display):   {app.service_principal_name}")
        break
    except Exception:
        if attempt < 4:
            time.sleep(5)
        else:
            raise RuntimeError(f"App '{APP_NAME}' not found after deploy.")

# Find synced-table pipelines and grant permissions
granted = 0
for p in w.pipelines.list_pipelines(filter=f"name LIKE '%{CATALOG}%'"):
    try:
        w.pipelines.update_permissions(
            pipeline_id=p.pipeline_id,
            access_control_list=[
                PipelineAccessControlRequest(
                    service_principal_name=sp_name,
                    permission_level=PipelinePermissionLevel.CAN_RUN
                )
            ]
        )
        granted += 1
        print(f"  ✅ {p.pipeline_id} | {p.name}")
    except Exception as e:
        print(f"  ⚠️  {p.pipeline_id}: {e}")

print(f"\n✅ Granted CAN_MANAGE_RUN on {granted} pipeline(s) to {sp_name}")

App SP (client_id): 4713da01-550f-40fa-9645-4c11a01beb29
App SP (display):   app-1s0sa2 lakebase-101-app
  ✅ 0f01e463-2da2-4b3f-8afe-15c800525c99 | Synced table: lakebase_101_catalog.lakebase_101_schema.customers_directory_synced HStSjQ
  ✅ 7c463bee-33b9-4030-9fc1-a4f9ced3b6db | Synced table: lakebase_101_catalog.lakebase_101_schema.sales_events_synced xZh477
  ✅ 8056f305-9c07-43d6-b8b3-222ba191dfd1 | Synced table: lakebase_101_catalog.lakebase_101_schema.customer_360_synced PaGOmL

✅ Granted CAN_MANAGE_RUN on 3 pipeline(s) to 4713da01-550f-40fa-9645-4c11a01beb29
